# 045 — Series temporales y backtesting

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Estructura:** `y_t = T_t + S_t + e_t` (o multiplicativa; log la vuelve aditiva).
**Estacionariedad** (media/varianza/autocovarianza constantes) es el supuesto de los
modelos clásicos; se alcanza diferenciando (`y_t − y_{t−1}`; estacional `y_t − y_{t−s}`).

**ARIMA(p,d,q):** AR = regresión sobre p rezagos propios; I = d diferencias; MA = q
errores pasados. ARIMA(0,1,0) = paseo aleatorio = baseline naive (`ŷ_t = y_{t−1}`).
Metodología Box-Jenkins: identificar con ACF/PACF → estimar → residuos ruido blanco.

**Backtesting (rolling origin):** entrenar con el pasado, predecir el bloque siguiente,
avanzar el origen y repetir (ventana expansiva o deslizante). Nunca split aleatorio:
fuga temporal. Features (rezagos, medias móviles) calculadas solo con datos ≤ t.

**Métrica honesta:** MAE/RMSE por horizonte + skill contra naive:
`skill = 1 − MAE_modelo/MAE_naive`. Sin superar al naive, no hay modelo.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (0-based, prediciendo índices 3..6). Naive: |11−9| = 2, |10−11| = 1,
|12−10| = 2, |11−12| = 1 → MAE = 6/4 = **1.50**. MM3: media(8,10,9) = 9 → |11−9| = 2;
media(10,9,11) = 10 → |10−10| = 0; media(9,11,10) = 10 → |12−10| = 2;
media(11,10,12) = 11 → |11−11| = 0 → MAE = 4/4 = **1.00**.
Skill = 1 − 1.00/1.50 ≈ **0.33**. La serie alterna (autocorrelación negativa a rezago 1):
el naive persigue el zigzag con un paso de retraso; la media móvil promedia el vaivén.

**Ejercicio 2.** (a) No: la media crece con t (tendencia lineal determinista).
(b) y' = [3, 3, 3, 3, 3]: constante (estacionaria y sin varianza).
(c) y_t = y_{t−1} + 3: paseo determinista con deriva; en la familia, un
**ARIMA(0,1,0) con constante** (la primera diferencia es c + ruido, aquí con ruido 0).

**Ejercicio 3.** Fuga 1 — *feature con futuro*: la media centrada usa 3 días posteriores
al día que describe; en predicción real no existen. Corregir: ventana SOLO hacia atrás
(rolling, no centered). Fuga 2 — *split aleatorio*: filas de test quedan rodeadas de
vecinos temporales en train, que comparten la misma ventana móvil. Corregir: hold-out
temporal o rolling origin, con gap ≥ ancho de la ventana.

**Ejercicio 4.** Folds (meses 1-24): train [1-12] → test [13-15]; train [1-15] →
test [16-18]; train [1-18] → test [19-21]; train [1-21] → test [22-24]. Son
**4 folds**: tras el train inicial quedan 12 meses de datos y cada fold consume 3 de
test; el último termina en 24 porque no hay datos para un quinto bloque de test.


In [ ]:
result = run_lab("ml", seed=45)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — backtest naive vs. media móvil (h=1)
y = [8, 10, 9, 11, 10, 12, 11]

def backtest_h1(y, prediccion):
    errores = [abs(y[t] - prediccion(y, t)) for t in range(3, len(y))]
    return sum(errores) / len(errores)

mae_naive = backtest_h1(y, lambda s, t: s[t - 1])
mae_mm3 = backtest_h1(y, lambda s, t: sum(s[t - 3:t]) / 3)
print(f"MAE naive = {mae_naive:.2f}  MAE MM3 = {mae_mm3:.2f}  skill = {1 - mae_mm3 / mae_naive:.2f}")
# La MM3 gana porque la serie alterna: el naive siempre llega un paso tarde.


In [ ]:
# Ejercicios 2 y 4 — diferenciación y folds walk-forward
y = [5, 8, 11, 14, 17, 20]
dif = [b - a for a, b in zip(y, y[1:])]
print("primera diferencia:", dif, "→ constante: tendencia eliminada con d=1")

n_meses, train_inicial, horizonte = 24, 12, 3
folds = []
fin_train = train_inicial
while fin_train + horizonte <= n_meses:
    folds.append((1, fin_train, fin_train + 1, fin_train + horizonte))
    fin_train += horizonte
for tr_a, tr_b, te_a, te_b in folds:
    print(f"train [{tr_a}-{tr_b}] → test [{te_a}-{te_b}]")
print(f"total: {len(folds)} folds")


## Reflexión

1. El laboratorio selecciona su umbral usando todos los ejemplos a la vez, sin noción de
   orden temporal. Si esos ejemplos fueran mediciones consecutivas de un sensor, ¿qué
   haría inválida esa selección y cómo la reorganizarías en un esquema walk-forward?
2. ¿Por qué un R² alto prediciendo el nivel de una serie persistente no demuestra nada, y
   qué comparación lo sustituye?
3. Tu backtest da MAE 1.2 y en producción el error es 2.5 desde el primer día. Enumera
   tres causas ordenadas de la más probable a la menos, con el diagnóstico para cada una.
